In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

import pickle

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

from langchain_community.vectorstores import FAISS

from rank_bm25 import BM25Okapi

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1756/3619117544.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [5]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [6]:
# ============================================================
# VERIFY DATASET
# ============================================================

print(
    eval_df.columns.tolist()
)

['query', 'article', 'reference_summary']


In [7]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [9]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [11]:
# ============================================================
# LOAD CHUNKS
# ============================================================

with open(

    "../notebooks/all_chunks.pkl",

    "rb"

) as f:

    all_chunks = pickle.load(
        f
    )

print(
    "Chunks Loaded:",
    len(all_chunks)
)

Chunks Loaded: 160571


In [12]:
# ============================================================
# CREATE TOKENIZED CHUNKS
# ============================================================

tokenized_chunks = [

    chunk.lower().split()

    for chunk in all_chunks

]

print(
    "Tokenization Completed"
)

Tokenization Completed


In [13]:
# ============================================================
# LOAD BM25
# ============================================================

bm25 = BM25Okapi(
    tokenized_chunks
)

print(
    "BM25 Loaded Successfully"
)

BM25 Loaded Successfully


In [14]:
# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================

def reciprocal_rank_fusion(

    dense_docs,

    sparse_docs,

    k=60

):

    scores = {}

    for rank, doc in enumerate(

        [

            d.page_content

            for d in dense_docs

        ]

    ):

        scores[doc] = scores.get(

            doc,

            0

        ) + 1 / (

            k + rank + 1

        )

    for rank, doc in enumerate(
        sparse_docs
    ):

        scores[doc] = scores.get(

            doc,

            0

        ) + 1 / (

            k + rank + 1

        )

    return sorted(

        scores.items(),

        key=lambda x: x[1],

        reverse=True

    )

In [15]:
# ============================================================
# LOAD GPT4o
# ============================================================

llm = ChatOpenAI(

    model="gpt-4o",

    temperature=0

)

print(
    "GPT4o Loaded Successfully"
)

GPT4o Loaded Successfully


In [16]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [17]:
# ============================================================
# HYBRID GPT4o EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 500:
        break

    if idx % 50 == 0:

        print(
            f"Processing Row {idx}"
        )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # ========================================================
    # DENSE RETRIEVAL
    # ========================================================

    dense_docs = vector_store.similarity_search(

        query,

        k=10

    )

    # ========================================================
    # SPARSE RETRIEVAL
    # ========================================================

    tokenized_query = query.lower().split()

    sparse_indices = bm25.get_top_n(

        tokenized_query,

        list(
            range(
                len(all_chunks)
            )
        ),

        n=10

    )

    sparse_docs = [

        all_chunks[idx]

        for idx in sparse_indices

    ]

    # ========================================================
    # RRF
    # ========================================================

    fused_docs = reciprocal_rank_fusion(

        dense_docs,

        sparse_docs

    )

    top_docs = [

        doc[0]

        for doc in fused_docs[:5]

    ]

    # ========================================================
    # CONTEXT
    # ========================================================

    context = "\n\n".join(
        top_docs
    )

    # ========================================================
    # PROMPT
    # ========================================================

    prompt = f"""

    Use only the context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # ========================================================
    # GENERATE RESPONSE
    # ========================================================

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "pipeline":
        "Hybrid GPT4o",

        "model":
        "GPT4o",

        "retrieval_method":
        "Hybrid",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nHybrid GPT4o Evaluation Completed"
)

Processing Row 0
Processing Row 50
Processing Row 100
Processing Row 150
Processing Row 200
Processing Row 250
Processing Row 300
Processing Row 350
Processing Row 400
Processing Row 450

Hybrid GPT4o Evaluation Completed


In [18]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 500


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Hybrid GPT4o,GPT4o,Hybrid,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,Hybrid GPT4o,GPT4o,Hybrid,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Hybrid GPT4o,GPT4o,Hybrid,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,to win the World Cup.
3,Hybrid GPT4o,GPT4o,Hybrid,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"judge a president's legacy, emphasizing that i..."
4,Hybrid GPT4o,GPT4o,Hybrid,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,underage.


In [19]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

].head(10)

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,to win the World Cup.
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"judge a president's legacy, emphasizing that i..."
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,underage.
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,his face-to-face showdown with Jim Cramer on T...
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,Bahr Idriss Abu Garda faces charges related to...
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,"Yes, South Africa beat cohosts India in the Wo..."
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,open mind
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,squad on Sunday for their World Cup qualifier ...


In [20]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "hybrid_gpt4o_eval.csv",

    index=False

)

print(
    "Hybrid GPT4o Evaluation Results Saved Successfully"
)

Hybrid GPT4o Evaluation Results Saved Successfully


In [21]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "hybrid_gpt4o_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(500, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Hybrid GPT4o,GPT4o,Hybrid,solar observatory aims to provide better under...,solar observatory aims to provide better under...,the sun and its role in space weather events s...
1,Hybrid GPT4o,GPT4o,Hybrid,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Hybrid GPT4o,GPT4o,Hybrid,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,to win the World Cup.
3,Hybrid GPT4o,GPT4o,Hybrid,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"judge a president's legacy, emphasizing that i..."
4,Hybrid GPT4o,GPT4o,Hybrid,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,underage.
